# B2.4 · Budgets and stop conditions

**Function B — Product & Application Security → The Security Automation / Harness Engineer**  ·  *AI for Security*

---

**Risk.** Every loop needs a reason to stop that isn't "it finished".

**Control.** Step limits, goal timeouts, token ceilings, spend circuit breakers.

**This lab.** Bound a divergent loop four different ways.

| | |
|---|---|
| Open-source tooling | Python |
| Open-weight models | Llama 3.3 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("B2.4"))

Budgets and stop conditions are the only controls that work when everything else has failed, including the verifier.

In [ ]:
from cybercommons import loop

# a model that never converges — the normal failure, not an exotic one
spinner = lambda: loop.FakeModel(["still working on it"])

for steps in (1, 3, 10):
    tr = loop.run(spinner(), loop.no_verifier(), max_steps=steps)
    print(f"max_steps={steps:2d} → {len(tr.steps):2d} steps, stopped by {tr.stopped_by}")

tr = loop.run(spinner(), loop.no_verifier(), max_steps=10_000, max_seconds=0.05)
print(f"\ntime budget → {len(tr.steps)} steps, stopped by {tr.stopped_by}")

Both budgets are real stop conditions and they bound different things. Step budgets bound *cost*; time budgets bound *damage*, because a loop calling a fast tool can do a lot of steps in a second.

In [ ]:
from cybercommons import ir
print(ir.containment_race(agent_actions_per_min=600, human_approval_minutes=5))

### Expect

Each step budget is respected exactly. The time budget stops the loop well short of 10,000 steps. The containment race shows ~3000 actions during a five-minute human approval.

### Your turn

What should happen at the stop — halt, or roll back? Those are different budgets. Write the rollback condition for one tool in your harness (B2.9 is the follow-through).

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/B2.4.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*